# 014 — Búsqueda en anchura y profundidad

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

La solución valida el contrato mínimo sin asumir un valor interno específico.


In [ ]:
result = run_lab("search", seed=14)
assert result["kind"] == "search"
assert result["evidence"]
show(result)


## Solución 1 — DFS a mano

Pila tras expandir `A`: `[B, C]`; sale `C` (el último apilado).

```text
expande A → apila B, C
expande C → apila F        pila: [B, F]
expande F → apila G        pila: [B, G]
expande G → objetivo ✔
```

a) Orden de expansión: `A, C, F, G`. b) Camino: `A→C→F→G` (3 aristas).
c) Coincide con BFS **por casualidad**: en este grafo los dos caminos a `G`
miden lo mismo. DFS no ofrece ninguna garantía de optimalidad; si `C→F→G`
midiera 10 aristas, DFS lo devolvería igual.


## Solución 2 — BFS a mano

Cola FIFO: `A` (encola B,C) → `B` (D,E) → `C` (F) → `D` → `E` (G) → `F` (G) → `G`.
Orden: `[A, B, C, D, E, F, G]`; camino `A→B→E→G`, costo 3 — exactamente lo que
reporta el laboratorio.


In [ ]:
result = run_lab("search", seed=14)
assert result["result"]["expanded"] == ["A", "B", "C", "D", "E", "F", "G"]
assert result["result"]["path"] == ["A", "B", "E", "G"]
print("traza BFS verificada ✔")


## Solución 3 — Memoria

a) `3^5 = 243` nodos de orden de magnitud para BFS (y crece exponencialmente
con `d`). b) `3 × 5 = 15` para DFS. c) Con `m = 10·d`, DFS puede perderse en
ramas profundas irrelevantes: IDS o BFS son preferibles si la memoria lo
permite. Con ciclos, DFS sin control de visitados **no termina**; cualquier
elección exige detección de estados repetidos.


In [ ]:
b, d, m = 3, 5, 5
frontera_bfs = b**d
frontera_dfs = b*m
print(f"BFS ~{frontera_bfs} nodos vs DFS ~{frontera_dfs} nodos")


## Solución 4 — Ciclo D → A

Un DFS sin visitados entraría en el bucle `A → B → D → A → B → D → ...` y no
terminaría jamás: la pila repetiría estados indefinidamente. BFS con `visited`
(como el laboratorio) descarta la re-expansión de `A` al reencontrarlo, así que
el ciclo solo añade una arista inútil. Moral: en grafos (a diferencia de
árboles), el control de repetidos no es una optimización, es corrección.


## Reflexión

1. En el grafo del laboratorio BFS y DFS devuelven caminos de la misma longitud. Dibuja una modificación mínima del grafo donde DFS devuelva un camino estrictamente peor.
2. BFS necesita memoria O(b^d). Con b=10 y d=10, ¿cuántos nodos serían? ¿Qué implica para problemas reales y por qué IDS es el compromiso estándar?
3. El laboratorio marca los nodos visitados al expandirlos. ¿Qué pasaría con la traza (y con la terminación) si no existiera el conjunto `visited` en un grafo con ciclos?
